In [2]:
# Function 2 - BBO Capstone Project
# Strategy: Gaussian Process + EI/UCB Hybrid
# Function 2 is a 2D black-box optimization problem.

import numpy as np
from scipy.stats import norm, qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


# --------------------------------------------------
# 1. Function 2 data
# --------------------------------------------------

X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.77862750],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.92656420],

    # Iteration results from our project
    [0.07344000, 0.99596600],
    [0.28663700, 0.86381700],
    [0.76404438, 0.93734368],
    [0.69715600, 0.04627700],
    [0.71353400, 0.62298300],
    [0.86021400, 0.47262600],
    [0.68231100, 0.70007500],
    [0.86539800, 0.16749100],
    [0.87632200, 0.45187500],
    [0.75333000, 0.03081900],
    [0.84048400, 0.36678800],
    [0.69336900, 0.31630400],
])

y = np.array([
    0.53899612,
    0.42058624,
    -0.06562362,
    0.29399291,
    0.21496451,
    0.02310555,
    0.24461934,
    0.03874902,
    -0.01385762,
    0.61120522,

    # Iteration outputs from our project
    0.545876278176769,
    0.064657824682165,
    0.175664369696048,
    0.661273278850939,
    0.5281368705482168,
    0.7843694604014655,
    0.571599983683343,
    0.4183161117651154,
    0.6773369360993695,
    0.294629069692715,
    0.3762580777463512,
    0.5698036470501392,
])


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def format_query(point, digits=6):
    """Format point as required by the BBO portal."""
    return "-".join(f"{value:.{digits}f}" for value in point)


def clip_to_bounds(points):
    """Keep all candidate values inside [0, 1)."""
    return np.clip(points, 0.0, 0.999999)


def expected_improvement(candidates, model, y_best, xi=0.01):
    """Expected Improvement acquisition function."""
    mean, std = model.predict(candidates, return_std=True)
    std = np.maximum(std, 1e-12)

    improvement = mean - y_best - xi
    z = improvement / std

    ei = improvement * norm.cdf(z) + std * norm.pdf(z)
    return ei


def upper_confidence_bound(candidates, model, kappa=2.0):
    """Upper Confidence Bound acquisition function."""
    mean, std = model.predict(candidates, return_std=True)
    return mean + kappa * std


# --------------------------------------------------
# 3. Check data and current best result
# --------------------------------------------------

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

best_index = np.argmax(y)
best_point = X[best_index]
best_y = y[best_index]

print("\nBest point so far:")
print(format_query(best_point))
print("Best output so far:", best_y)


# --------------------------------------------------
# 4. Gaussian Process setup
# --------------------------------------------------

kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2],
        length_scale_bounds=(1e-3, 1e3),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-2)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    alpha=1e-8,
    n_restarts_optimizer=20,
    random_state=42
)

gpr.fit(X, y)

print("\nGP Model Diagnostics:")
print("Kernel:", gpr.kernel_)
print("Training score:", round(gpr.score(X, y), 4))


# --------------------------------------------------
# 5. Generate candidate points
# --------------------------------------------------
# Strategy:
# 60% local exploitation around best point
# 40% global exploration using Latin Hypercube

np.random.seed(42)

local_candidates = best_point + np.random.normal(
    loc=0.0,
    scale=0.08,
    size=(3000, 2)
)

local_candidates = clip_to_bounds(local_candidates)

sampler = qmc.LatinHypercube(d=2, seed=42)
global_candidates = sampler.random(n=2000)

X_candidates = np.vstack([
    local_candidates,
    global_candidates
])


# --------------------------------------------------
# 6. Acquisition scoring: EI + UCB
# --------------------------------------------------

EI_XI = 0.01
UCB_KAPPA = 2.0

ei_scores = expected_improvement(
    candidates=X_candidates,
    model=gpr,
    y_best=best_y,
    xi=EI_XI
)

ucb_scores = upper_confidence_bound(
    candidates=X_candidates,
    model=gpr,
    kappa=UCB_KAPPA
)

ei_norm = (ei_scores - ei_scores.min()) / (ei_scores.max() - ei_scores.min() + 1e-12)
ucb_norm = (ucb_scores - ucb_scores.min()) / (ucb_scores.max() - ucb_scores.min() + 1e-12)

hybrid_score = 0.65 * ei_norm + 0.35 * ucb_norm


# --------------------------------------------------
# 7. Select next query
# --------------------------------------------------

best_candidate_index = np.argmax(hybrid_score)
x_next = X_candidates[best_candidate_index]

predicted_y, predicted_std = gpr.predict(
    x_next.reshape(1, -1),
    return_std=True
)

print("\nNext Point to Sample:")
print("X =", x_next)
print("Submission format:", format_query(x_next))
print("Predicted y:", predicted_y[0])
print("Uncertainty:", predicted_std[0])
print("Hybrid score:", hybrid_score[best_candidate_index])


# --------------------------------------------------
# 8. Show top 5 candidate points
# --------------------------------------------------

top5_index = np.argsort(hybrid_score)[-5:][::-1]

print("\nTop 5 Candidates:")

for rank, idx in enumerate(top5_index, start=1):
    point = X_candidates[idx]
    mean_i, std_i = gpr.predict(point.reshape(1, -1), return_std=True)

    print(
        f"{rank}. X={format_query(point)} | "
        f"pred={mean_i[0]:.6f}, "
        f"std={std_i[0]:.6f}, "
        f"score={hybrid_score[idx]:.6f}"
    )

Shape of X: (22, 2)
Shape of y: (22,)

Best point so far:
0.860214-0.472626
Best output so far: 0.7843694604014655

GP Model Diagnostics:
Kernel: 0.95**2 * Matern(length_scale=[0.0229, 0.317], nu=2.5) + WhiteKernel(noise_level=8.45e-10)
Training score: 1.0

Next Point to Sample:
X = [0.86500861 0.49481616]
Submission format: 0.865009-0.494816
Predicted y: 0.7946858087726454
Uncertainty: 0.03685623968065912
Hybrid score: 0.9949456706273541

Top 5 Candidates:
1. X=0.865009-0.494816 | pred=0.794686, std=0.036856, score=0.994946
2. X=0.864775-0.494113 | pred=0.795431, std=0.035596, score=0.988795
3. X=0.865021-0.503691 | pred=0.791468, std=0.039991, score=0.982270
4. X=0.864933-0.508376 | pred=0.789549, std=0.041477, score=0.969708
5. X=0.865573-0.513901 | pred=0.784924, std=0.045999, score=0.961169
